# 16.2.2 Reusing Pretrained Embeddings

## Dataset

In [1]:
import tensorflow as tf
from typing import Tuple

2023-04-06 07:27:27.829114: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import tensorflow_datasets as tfds

datasets, info = tfds.load("imdb_reviews", as_supervised=True, with_info=True)

2023-04-06 07:27:30.222182: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-04-06 07:27:30.329514: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-04-06 07:27:30.329745: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-04-06 07:27:30.330937: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropri

## Preprocessing

In [3]:
train_set = datasets["train"].batch(32).prefetch(1)

## Model

In [6]:
import tensorflow as tf
from tensorflow_hub import KerasLayer

model = tf.keras.models.Sequential(
    [
        KerasLayer(
            "http://tfhub.dev/google/tf2-preview/nnlm-en-dim50/1",
            dtype=tf.string,
            input_shape=[],
            output_shape=[50],
        ),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ]
)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 keras_layer_2 (KerasLayer)  (None, 50)                48190600  
                                                                 
 dense_4 (Dense)             (None, 128)               6528      
                                                                 
 dense_5 (Dense)             (None, 1)                 129       
                                                                 
Total params: 48,197,257
Trainable params: 6,657
Non-trainable params: 48,190,600
_________________________________________________________________


In [7]:
import time
from pathlib import Path

root_logdir = Path().absolute() / "logs"

%load_ext tensorboard
%tensorboard --logdir=./logs --port=6006

Launching TensorBoard...

In [8]:
log_dir = root_logdir / time.strftime("run_%Y_%m_%d-%H_%M_%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir, histogram_freq=1)

In [9]:
history = model.fit(train_set, epochs=5, callbacks=[tensorboard_callback])

Epoch 1/5
771/782 [============================>.] - ETA: 0s - loss: 0.5452 - accuracy: 0.7264

2023-04-06 07:29:03.140447: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 11565744000 exceeds 10% of free system memory.


782/782 [==============================] - 9s 10ms/step - loss: 0.5449 - accuracy: 0.7266
Epoch 2/5
779/782 [============================>.] - ETA: 0s - loss: 0.5131 - accuracy: 0.7485

2023-04-06 07:29:11.044034: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 11565744000 exceeds 10% of free system memory.


782/782 [==============================] - 7s 9ms/step - loss: 0.5132 - accuracy: 0.7485
Epoch 3/5
772/782 [============================>.] - ETA: 0s - loss: 0.5084 - accuracy: 0.7514

...

782/782 [==============================] - 7s 9ms/step - loss: 0.5087 - accuracy: 0.7514
Epoch 4/5
775/782 [============================>.] - ETA: 0s - loss: 0.5054 - accuracy: 0.7532

2023-04-06 07:29:25.308347: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 11565744000 exceeds 10% of free system memory.


782/782 [==============================] - 8s 10ms/step - loss: 0.5055 - accuracy: 0.7531
Epoch 5/5
781/782 [============================>.] - ETA: 0s - loss: 0.5029 - accuracy: 0.7548

2023-04-06 07:29:32.782315: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 11565744000 exceeds 10% of free system memory.


782/782 [==============================] - 7s 9ms/step - loss: 0.5030 - accuracy: 0.7548


In [10]:
test_set = datasets["test"].batch(32).prefetch(1)
model.evaluate(test_set)

782/782 [==============================] - 3s 4ms/step - loss: 0.5125 - accuracy: 0.7468


[0.5125075578689575, 0.7468000054359436]